# Réanalyse des variables et modèles de gravité

Les cellules lisent les résultats déjà calculés ; elles ne relancent pas la recherche. Les sinistres extrêmes sont conservés et le notebook original reste inchangé. Les associations étudiées après consultation des données sont descriptives, pas causales.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT=next(p for p in [Path.cwd(),Path.cwd().parent] if (p/'run.py').exists())
R=ROOT/'reports/variable_models'
r=json.loads((R/'results.json').read_text())
receipt=json.loads((R/'kaggle_result.json').read_text())
receipt

{'file': 'deliverables\\candidates\\submission_variable_models.csv',
 'sha256': '70a369b8201fd6044b0938cdfb21e1bd2ac501d9fe85ad6972cb584f18481aeb',
 'public_score': 1365.745,
 'source': 'User supplied Kaggle Complete public score',
 'independently_browser_verified': False,
 'local_date': '2026-10-12',
 'previous_best_public_score': 1365.913,
 'improvement': 0.168,
 'target': 1365.5,
 'target_reached': False}

In [2]:
pd.read_csv(ROOT/'reports/restart_xgboost/variable_dictionary.csv')

,variable,meaning,role,train_missing,train_unique,test_missing,test_unseen,minimum,maximum,train_test_KS
0,index,Submission row identifier,excluded / target / grouping,0,50000,0.0,NaN,0.0,49999.00,1.000000
1,id_client,Client grouping identifier,excluded / target / grouping,0,45785,NaN,NaN,NaN,NaN,NaN
2,id_vehicule,Vehicle identifier,excluded / target / grouping,0,5,NaN,NaN,NaN,NaN,NaN
3,id_contrat,Contract identifier,excluded / target / grouping,0,50000,NaN,NaN,NaN,NaN,NaN
4,bonus,Bonus-malus coefficient,predictor,0,73,0.0,NaN,0.5,2.16,0.005340
5,type_contrat,Coverage level,predictor,0,4,0.0,0.0,NaN,NaN,NaN
6,duree_contrat,Contract duration; unit unconfirmed,predictor,0,41,0.0,NaN,1.0,41.00,0.002980
7,anciennete_info,Information tenure; unit unconfirmed,predictor,0,25,0.0,NaN,1.0,25.00,0.003020
8,freq_paiement,Payment frequency,predictor,0,4,0.0,0.0,NaN,NaN,NaN
9,paiement,Payment method,predictor,0,2,0.0,0.0,NaN,NaN,NaN


In [3]:
g=pd.read_csv(ROOT/'reports/variable_reanalysis/stable_descriptive_severity_groups.csv')
g[['variable','level','claims','weighted_severity_relative_residual','top25_claims','severity_residual_without_top25_diagnostic']].head(15)

,variable,level,claims,weighted_severity_relative_residual,top25_claims,severity_residual_without_top25_diagnostic
0,marque_vehicule,MERCEDES BENZ,118,0.222560,4,0.055784
1,modele_vehicule,206,123,0.175045,1,0.126120
2,licence_start_age,"(26.0, 69.0]",461,0.112322,5,0.014558
3,marque_vehicule,__OTHER__,127,-0.101452,0,-0.101452
4,modele_vehicule,CLIO,192,0.096550,2,0.066970
5,debut_vente_vehicule,"(13.0, 18.0]",222,-0.086827,0,-0.086827
6,type_contrat,Median2,283,-0.084053,1,-0.094619
7,marque_vehicule,TOYOTA,108,-0.082064,0,-0.082064
8,type_vehicule,Commercial,197,-0.064661,0,-0.064661
9,age_conducteur1,"(54.0, 62.0]",533,-0.064002,2,-0.091115


Les 25 coûts les plus élevés sont retirés uniquement dans la colonne diagnostique d'influence. Leur retrait réduit fortement certains effets apparents. Ils restent dans tous les entraînements. Catégories rares, permis incohérents et mesures nulles demandent une forte régularisation. L'absence du second conducteur est représentée séparément de ses mesures manquantes.

20 configurations, 61 choix : courbes splines Ridge sur familles véhicule/conducteur/contrat/ensemble avec quatre régularisations ; quatre XGBoost sur toutes les variables ; mélanges à 25/50/100 % avec le modèle précédent, qui reste candidat seul. Trois plis externes et cinq plis internes séparés par client. Toutes les transformations et tous les ajustements sont appris dans les plis d'apprentissage. Les données ont déjà été explorées ; cette évaluation n'est pas un nouveau test indépendant.

In [4]:
pd.DataFrame({name:r[name] for name in ['selected','incumbent']})

,selected,incumbent
MSE,293702.110276,293797.307272
RMSE,541.942903,542.030725
MAE,190.660512,191.035702
bias,-0.444843,-0.094337
predicted_observed_ratio,0.995697,0.999087


In [5]:
pd.DataFrame([{'fold':x['fold'],'selected_RMSE':x['selected']['RMSE'],'incumbent_RMSE':x['incumbent']['RMSE'],'choice':x['choice']} for x in r['folds']])

,fold,selected_RMSE,incumbent_RMSE,choice
0,1,569.500684,569.519671,"{'candidate': 13, 'weight': 0.5, 'selection_RM..."
1,2,517.902117,518.123563,"{'candidate': 18, 'weight': 1.0, 'selection_RM..."
2,3,537.170727,537.202874,"{'candidate': 12, 'weight': 0.5, 'selection_RM..."


In [6]:
r['gain']

{'gain_RMSE': 0.08782224270123606,
 'interval_95': [-0.02903266884801212, 0.2086924317671446],
 'repetitions': 2000,
 'interpretation': 'Descriptive paired client bootstrap conditional on fitted predictions; not full training uncertainty.'}

In [7]:
s=json.loads((R/'full_training/selection.json').read_text())
print('Poids :',r['final_choice']['weight'])
s['catalogue'][r['final_choice']['candidate']]

Poids : 0.25


{'kind': 'ridge', 'panel': 'all', 'alpha': 10.0}

In [8]:
v=json.loads((R/'verification.json').read_text())
assert v['frequency_unchanged'] and v['fresh_process_byte_identical']
v

{'status': 'verified',
 'rows': 50000,
 'tests_passed': 46,
 'fresh_process_byte_identical': True,
 'frequency_unchanged': True,
 'original_notebook_sha256': '76248bdb51400fe399c670a9b71092cabe041cc13d3839b83846ac1499938429',
 'candidate_sha256': '70a369b8201fd6044b0938cdfb21e1bd2ac501d9fe85ad6972cb584f18481aeb',
 'model_sha256': '9dc910e39b3a791092a3cadf93a5af886a78889949b195539e8e094782b8e555',
 'script_sha256': '3f7c54dbb3ca6c12628cd5992a645dc45e3b307639d6384900fee75ba6797abc',
 'configuration': {'kind': 'ridge', 'panel': 'all', 'alpha': 10.0},
 'provenance': {'data': {'train.csv': '8b5ca8a59c214269056ed1720de33619c10968549c383ec350898a659b97bfc5',
   'test.csv': '68cdf87201974688d754d846c2d8ff33c5aa2f9c011e84fba92018186cd8f569'},
  'source': {'__init__.py': '100b3e376bb5c6d141565df62567e20d231861a23f17cb8e15d587cce02ee61e',
   'experiment.py': '0237a7515b456a562c85c8b5e7339f2d08ca5a179c75bb7c7da33f6c86522eab',
   'models.py': 'a2787421cff97fa7ca2f7fb21d863eec035c61cd6826f459bbcb476

La gravité finale mélange 25 % du modèle Ridge sur variables étendues et 75 % du précédent meilleur modèle. La fréquence originale reste inchangée. Le score public transmis est 1365.745, contre 1365.913 auparavant. L'intervalle local inclut zéro ; le score public ne prouve pas une supériorité sur le test privé. L'objectif strictement inférieur à 1365.500 reste non atteint. Commandes et détails : `docs/variable_reanalysis.md`.